# Day 06 — Mock Interview Practice

## Difficulty Level: BASIC → INTERMEDIATE (Week 1, Day 6)

### Today's Focus
Building strong foundations across 8 core categories with 18 interview questions. Each day builds progressively on previous topics.

### Categories
| # | Category | Questions | Difficulty |
|---|----------|-----------|------------|
| 1 | SQL Advanced | 2 | BASIC, INTERMEDIATE |
| 2 | Python — Loop/Function/List/Dict | 2 | BASIC, INTERMEDIATE |
| 3 | PySpark Coding | 2 | BASIC, INTERMEDIATE |
| 4 | Spark Optimization & Deep Spark | 2 | BASIC, INTERMEDIATE |
| 5 | ETL — Failures, Late Data, Schema Handling | 3 | BASIC, INTERMEDIATE |
| 6 | Data Modeling | 2 | BASIC, INTERMEDIATE |
| 7 | Snowflake | 2 | BASIC, INTERMEDIATE |
| 8 | Monitoring, Logging & Orchestration | 3 | BASIC, INTERMEDIATE |
| 9 | Production Pillars — Security & Backfills (Project-Based) | 2 | INTERMEDIATE |

**Total: 20 Questions | 9 Categories**

## Q1: SQL Advanced — ROW_NUMBER for Deduplication [BASIC]

### Problem
You have a `customer_orders` table with duplicate entries — the same `order_id` appears multiple times due to retries. Write a SQL query to remove duplicates by keeping only the most recent record per `order_id` based on the `updated_at` timestamp.

### Sample Data
| order_id | customer_id | amount | updated_at |
|----------|-------------|--------|------------|
| 1001 | 501 | 150.00 | 2026-01-15 10:00:00 |
| 1001 | 501 | 175.50 | 2026-01-15 12:30:00 |
| 1002 | 502 | 200.00 | 2026-01-15 09:00:00 |
| 1003 | 503 | 350.00 | 2026-01-15 11:00:00 |
| 1003 | 503 | 325.00 | 2026-01-15 08:00:00 |

### Expected Output
| order_id | customer_id | amount | updated_at |
|----------|-------------|--------|------------|
| 1001 | 501 | 175.50 | 2026-01-15 12:30:00 |
| 1002 | 502 | 200.00 | 2026-01-15 09:00:00 |
| 1003 | 503 | 350.00 | 2026-01-15 11:00:00 |

In [0]:
from pyspark.sql import Row

# Create sample data
data = [
    Row(order_id=1001, customer_id=501, amount=150.00, updated_at="2026-01-15 10:00:00"),
    Row(order_id=1001, customer_id=501, amount=175.50, updated_at="2026-01-15 12:30:00"),
    Row(order_id=1002, customer_id=502, amount=200.00, updated_at="2026-01-15 09:00:00"),
    Row(order_id=1003, customer_id=503, amount=350.00, updated_at="2026-01-15 11:00:00"),
    Row(order_id=1003, customer_id=503, amount=325.00, updated_at="2026-01-15 08:00:00"),
]

df = spark.createDataFrame(data)
df.createOrReplaceTempView("customer_orders")

# Deduplicate using ROW_NUMBER() — keep latest record per order_id
result = spark.sql("""
    SELECT order_id, customer_id, amount, updated_at
    FROM (
        SELECT 
            order_id, customer_id, amount, updated_at,
            ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC) as rn
        FROM customer_orders
    )
    WHERE rn = 1
    ORDER BY order_id
""")

display(result)

## Q2: SQL Advanced — Cumulative Sum / Running Total [INTERMEDIATE]

### Problem
Given a `daily_sales` table with `date`, `store_id`, and `revenue` columns, calculate:
1. A running total (cumulative sum) of revenue for each store, ordered by date
2. The percentage contribution of each day's revenue to the store's running total so far

### Sample Data
| date | store_id | revenue |
|------|----------|---------|
| 2026-01-01 | S1 | 1000 |
| 2026-01-01 | S2 | 800 |
| 2026-01-02 | S1 | 1200 |
| 2026-01-02 | S2 | 600 |
| 2026-01-03 | S1 | 1500 |
| 2026-01-03 | S2 | 900 |

### Expected Output
| date | store_id | revenue | running_total | pct_of_running |
|------|----------|---------|----------------|----------------|
| 2026-01-01 | S1 | 1000 | 1000 | 100.00 |
| 2026-01-02 | S1 | 1200 | 2200 | 54.55 |
| 2026-01-03 | S1 | 1500 | 3700 | 40.54 |

In [0]:
from pyspark.sql import Row

data = [
    Row(date="2026-01-01", store_id="S1", revenue=1000),
    Row(date="2026-01-01", store_id="S2", revenue=800),
    Row(date="2026-01-02", store_id="S1", revenue=1200),
    Row(date="2026-01-02", store_id="S2", revenue=600),
    Row(date="2026-01-03", store_id="S1", revenue=1500),
    Row(date="2026-01-03", store_id="S2", revenue=900),
]

df = spark.createDataFrame(data)
df.createOrReplaceTempView("daily_sales")

result = spark.sql("""
    SELECT 
        date,
        store_id,
        revenue,
        SUM(revenue) OVER (
            PARTITION BY store_id 
            ORDER BY date 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_total,
        ROUND(
            revenue * 100.0 / 
            SUM(revenue) OVER (
                PARTITION BY store_id 
                ORDER BY date 
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ), 2
        ) AS pct_of_running_total
    FROM daily_sales
    ORDER BY store_id, date
""")

display(result)

## Q3: Python — Two Sum [BASIC]

### Problem
Given an array of integers and a target value, find the indices of two numbers that add up to the target. Use a hash map (dictionary) approach for O(n) time complexity. If no pair exists, return an empty list.

### Example
```python
nums = [2, 7, 11, 15], target = 9
# Output: [0, 1]  (because nums[0] + nums[1] = 2 + 7 = 9)

nums = [3, 2, 4], target = 6
# Output: [1, 2]  (because nums[1] + nums[2] = 2 + 4 = 6)
```

In [0]:
def two_sum(nums, target):
    """Find indices of two numbers that sum to target using hash map.
    Time: O(n), Space: O(n)
    """
    seen = {}  # value -> index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

# Test cases
assert two_sum([2, 7, 11, 15], 9) == [0, 1], "Test 1 failed"
assert two_sum([3, 2, 4], 6) == [1, 2], "Test 2 failed"
assert two_sum([3, 3], 6) == [0, 1], "Test 3 failed"
assert two_sum([1, 2, 3], 7) == [], "Test 4 failed"
assert two_sum([5, 5, 5, 5], 10) == [0, 1], "Test 5 failed"

print("All test cases passed!")
print(f"two_sum([2, 7, 11, 15], 9) = {two_sum([2, 7, 11, 15], 9)}")
print(f"two_sum([3, 2, 4], 6) = {two_sum([3, 2, 4], 6)}")

## Q4: Python — Flatten Nested List [INTERMEDIATE]

### Problem
Write a function to flatten a deeply nested list structure. For example:
```python
[1, [2, [3, 4], 5], 6, [7, [8, [9]]]]
# Should return: [1, 2, 3, 4, 5, 6, 7, 8, 9]
```

Provide both:
1. A recursive solution
2. An iterative solution using a stack

Also handle edge cases: empty lists, deeply nested single elements, and non-list iterables (strings should NOT be expanded).

In [0]:
def flatten_recursive(nested):
    """Recursively flatten a nested list."""
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(flatten_recursive(item))
        else:
            result.append(item)
    return result

def flatten_iterative(nested):
    """Iteratively flatten a nested list using a stack."""
    result = []
    stack = list(nested)  # copy to avoid modifying input
    while stack:
        item = stack.pop(0)  # pop from front (BFS-like)
        if isinstance(item, list):
            # Push items back to front in reverse order to maintain order
            for sub in reversed(item):
                stack.insert(0, sub)
        else:
            result.append(item)
    return result

# Test cases
nested = [1, [2, [3, 4], 5], 6, [7, [8, [9]]]]
expected = [1, 2, 3, 4, 5, 6, 7, 8, 9]

assert flatten_recursive(nested) == expected, f"Recursive failed: {flatten_recursive(nested)}"
assert flatten_iterative(nested) == expected, f"Iterative failed: {flatten_iterative(nested)}"

# Edge cases
assert flatten_recursive([]) == []
assert flatten_iterative([]) == []
assert flatten_recursive([[[1]]]) == [1]
assert flatten_iterative([[[1]]]) == [1]
assert flatten_recursive(["hello", ["world", ["!"]]]) == ["hello", "world", "!"]
assert flatten_iterative(["hello", ["world", ["!"]]]) == ["hello", "world", "!"]

print("All test cases passed!")
print(f"Recursive: {flatten_recursive(nested)}")
print(f"Iterative: {flatten_iterative(nested)}")

## Q5: PySpark — Filter, Select, and Aggregate [BASIC]

### Problem
Given an employee DataFrame with columns `emp_id`, `name`, `department`, `salary`, and `hire_date`, perform the following operations:
1. Filter employees with salary > 50000
2. Select only `name` and `department` columns from the filtered results
3. Group by department and count employees, plus calculate average salary

### Sample Data
| emp_id | name | department | salary | hire_date |
|--------|------|------------|--------|-----------|
| 1 | Alice | Engineering | 75000 | 2023-01-15 |
| 2 | Bob | Sales | 45000 | 2023-03-20 |
| 3 | Carol | Engineering | 85000 | 2022-11-10 |
| 4 | Dave | Sales | 55000 | 2023-06-01 |
| 5 | Eve | Engineering | 60000 | 2023-07-15 |

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, avg, count, round as _round

# Create sample data
employees_data = [
    Row(emp_id=1, name="Alice", department="Engineering", salary=75000, hire_date="2023-01-15"),
    Row(emp_id=2, name="Bob", department="Sales", salary=45000, hire_date="2023-03-20"),
    Row(emp_id=3, name="Carol", department="Engineering", salary=85000, hire_date="2022-11-10"),
    Row(emp_id=4, name="Dave", department="Sales", salary=55000, hire_date="2023-06-01"),
    Row(emp_id=5, name="Eve", department="Engineering", salary=60000, hire_date="2023-07-15"),
]

emp_df = spark.createDataFrame(employees_data)

# 1. Filter employees with salary > 50000
high_earners = emp_df.filter(col("salary") > 50000)
print("1. Employees with salary > 50000:")
display(high_earners)

# 2. Select only name and department
name_dept = high_earners.select("name", "department")
print("2. Name and Department only:")
display(name_dept)

# 3. Group by department: count + average salary
dept_stats = emp_df.groupBy("department").agg(
    count("*").alias("emp_count"),
    _round(avg("salary"), 2).alias("avg_salary")
).orderBy("department")
print("3. Department-wise stats:")
display(dept_stats)

## Q6: PySpark — Multiple Join Types [INTERMEDIATE]

### Problem
Given two DataFrames:
- `employees` (emp_id, name, dept_id)
- `departments` (dept_id, dept_name)

Demonstrate the following join types and explain the differences in results:
1. Inner join — only matching records
2. Left join — all employees, even without a department
3. Full outer join — all records from both sides

### Sample Data
**employees:**
| emp_id | name | dept_id |
|--------|------|---------|
| 1 | Alice | 10 |
| 2 | Bob | 20 |
| 3 | Carol | NULL |

**departments:**
| dept_id | dept_name |
|---------|-----------|
| 10 | Engineering |
| 20 | Sales |
| 30 | Marketing |

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col

employees = spark.createDataFrame([
    Row(emp_id=1, name="Alice", dept_id=10),
    Row(emp_id=2, name="Bob", dept_id=20),
    Row(emp_id=3, name="Carol", dept_id=None),
])

departments = spark.createDataFrame([
    Row(dept_id=10, dept_name="Engineering"),
    Row(dept_id=20, dept_name="Sales"),
    Row(dept_id=30, dept_name="Marketing"),
])

# 1. Inner join — only matching records
inner_result = employees.join(departments, on="dept_id", how="inner")
print("1. Inner Join:")
display(inner_result)

# 2. Left join — all employees, even without a department
left_result = employees.join(departments, on="dept_id", how="left")
print("2. Left Join:")
display(left_result)

# 3. Full outer join — all records from both sides
full_result = employees.join(departments, on="dept_id", how="full_outer")
print("3. Full Outer Join:")
display(full_result)

# Key differences:
# - Inner: Carol (no dept) excluded, Marketing (no emp) excluded → 2 rows
# - Left: Carol included with NULL dept_name, Marketing excluded → 3 rows
# - Full outer: Carol included, Marketing included → 4 rows

## Q7: Spark Optimization — Repartition vs Coalesce [BASIC]

### Problem
Explain the difference between `repartition()` and `coalesce()` in Apache Spark:
1. What does each operation do?
2. When does a full shuffle occur vs a local shuffle?
3. When would you use each one?
4. Show a code example demonstrating the effect on partition count.

### Key Concepts
- `repartition(n)`: Full shuffle, can increase or decrease partitions, creates equal-sized partitions
- `coalesce(n)`: Minimizes data movement (local shuffle only), can only decrease partitions, may create uneven partition sizes
- Use `coalesce` when reducing partitions (e.g., before writing to a single file)
- Use `repartition` when increasing partitions or when you need even distribution

In [0]:
# Create a DataFrame with some data
df = spark.range(0, 1000)
shuffle_parts = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Default shuffle partitions: {shuffle_parts}")

# repartition: full shuffle, creates exactly N partitions
df_repart = df.repartition(8)
print(f"After repartition(8): {df_repart.count()} rows across 8 partitions (full shuffle)")

# coalesce: local merge, reduces partitions without full shuffle
df_coalesce = df_repart.coalesce(2)
print(f"After coalesce(2): {df_coalesce.count()} rows across 2 partitions (local merge, no shuffle)")

# coalesce can ONLY reduce, not increase (without shuffle=True)
# If you try coalesce(16) from 8 partitions, it stays at 8
df_coalesce_up = df_repart.coalesce(16)
print(f"coalesce(16) from 8 partitions: stays at 8 partitions (can't increase without shuffle)")

# To increase partitions, use repartition (coalesce without shuffle can only decrease)
df_repartitioned = df_coalesce.repartition(16)
print(f"repartition(16) from 2 partitions: now 16 partitions (full shuffle)")
print("Note: coalesce() can only DECREASE partitions. To increase, use repartition().")

# Practical use case: before writing to storage, coalesce to fewer files
# df_repart.coalesce(1).write.mode("overwrite").csv("/Volumes/catalog/schema/volume/output")
print("\n--- Explanation ---")
print("repartition(n): Full shuffle across network. Use when increasing partitions or needing even distribution.")
print("coalesce(n): Local merge within nodes (no full shuffle). Use when reducing partitions, e.g., before writing output.")
print("coalesce(n, shuffle=True): Full shuffle like repartition but with optimized partition count.")
print("\nNote: df.rdd.getNumPartitions() is not available on serverless compute.")
print("Use spark.conf.get('spark.sql.shuffle.partitions') to check shuffle partition config.")

## Q8: Spark Optimization — File Format: Parquet vs Delta Lake [INTERMEDIATE]

### Problem
Compare Parquet and Delta Lake file formats for Spark data storage:
1. What are the key advantages of Delta over plain Parquet?
2. How does ACID transaction support work in Delta?
3. What is the transaction log (OPTIMIZE, ZORDER, TIME TRAVEL)?
4. Show code examples for writing and reading in both formats.

### Key Differences
| Feature | Parquet | Delta Lake |
|---------|---------|------------|
| ACID Transactions | No | Yes |
| Schema Evolution | Limited (mergeSchema) | Native |
| Time Travel | No | Yes (VERSION AS OF) |
| Upsert/Delete | No (rewrite files) | Yes (MERGE/UPDATE/DELETE) |
| Transaction Log | No | Yes (_delta_log) |
| Auto Compaction | No | Yes (OPTIMIZE) |
| ZORDER Clustering | No | Yes (ZORDER BY) |

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable

# Create sample data
data = [
    Row(id=1, name="Alice", city="NYC"),
    Row(id=2, name="Bob", city="LA"),
    Row(id=3, name="Carol", city="SF"),
]
df = spark.createDataFrame(data)

# --- Write as Delta (managed UC table) ---
# Parquet is the underlying file format for Delta — Delta adds the transaction layer
df.write.format("delta").mode("overwrite").saveAsTable("day06_q8_delta")

delta_df = spark.table("day06_q8_delta")
print("Delta table read:")
display(delta_df)

# --- Delta advantages over plain Parquet ---

# 1. UPDATE rows (not possible with plain Parquet)
spark.sql("""
    UPDATE day06_q8_delta 
    SET city = 'New York' 
    WHERE id = 1
""")

# 2. DELETE rows (not possible with plain Parquet)
spark.sql("""
    DELETE FROM day06_q8_delta 
    WHERE id = 3
""")

# 3. MERGE (upsert)
new_data = spark.createDataFrame([Row(id=4, name="Dave", city="Boston")])
new_data.write.format("delta").mode("append").saveAsTable("day06_q8_delta")

# 4. Time Travel — read previous version
old_version = spark.sql("SELECT * FROM day06_q8_delta VERSION AS OF 0")
print("Time Travel — version 0 (original data):")
display(old_version)

print("\n--- Key Delta Advantages ---")
print("1. ACID transactions via _delta_log transaction log")
print("2. UPDATE/DELETE/MERGE without rewriting entire dataset")
print("3. Time Travel: query previous versions with VERSION AS OF or TIMESTAMP AS OF")
print("4. Schema evolution with automatic mergeSchema")
print("5. OPTIMIZE compacts small files; ZORDER BY clusters data for faster queries")
print("6. Parquet is the underlying file format — Delta adds the transaction layer on top")

## Q9: ETL — Structured Streaming Checkpointing [BASIC]

### Problem
Explain what checkpointing does in Spark Structured Streaming:
1. What information is stored in the checkpoint directory?
2. Why is checkpointing essential for fault tolerance?
3. What happens if a streaming job restarts without a checkpoint?
4. Write a basic streaming query that reads from a rate source, uses a checkpoint location, and writes to a memory sink.

### Key Concepts
- **Checkpoint location**: A directory (usually on durable storage like S3/ADLS) where Spark saves:
  - Offset log: which data has been consumed
  - Commit log: which batches have been committed
  - Metadata: streaming query configuration and state
- Without checkpointing, a restarted job starts from scratch (reprocesses all data or misses data)
- With checkpointing, the job resumes from where it left off

In [0]:
# Structured Streaming checkpointing demonstration
# Note: Streaming with rate source + checkpoint requires file-based checkpoint location
# which is not available on serverless compute. This is an explanatory demo.

print("""
--- Structured Streaming Checkpointing Code (Production Example) ---

from pyspark.sql.functions import col

# Create a streaming query with checkpointing using rate source
streaming_df = (spark
    .readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

# Write to memory sink with checkpointing
# In production, checkpoint location should be on durable storage (S3/ADLS/GCS or UC Volume)
checkpoint_path = "/Volumes/catalog/schema/volume/checkpoints/rate_stream"

query = (streaming_df
    .writeStream
    .format("memory")
    .queryName("rate_stream_table")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(processingTime="5 seconds")
    .start()
)

# Let it run for a few seconds
import time
time.sleep(8)

# Check what's in the memory sink
result = spark.sql("SELECT * FROM rate_stream_table ORDER BY value DESC LIMIT 10")
display(result)

# Stop the stream
query.stop()

--- Checkpointing Summary ---

1. Checkpoint stores: offset log (consumed offsets), commit log (committed batches), metadata
2. If the job crashes and restarts, it resumes from the last committed offset
3. Without checkpoint: job starts from beginning (reprocesses or loses data)
4. Always use durable storage (S3/ADLS or UC Volume) for checkpoint location in production
5. Checkpoint location must be unique per streaming query — sharing causes corruption
6. Checkpoint contains:
   - offsets/ : which data has been consumed from source
   - commits/ : which batches have been committed
   - metadata : streaming query configuration
   - state/ : aggregated state for stateful operations
""")

## Q10: ETL — Idempotent Writes with Delta MERGE [INTERMEDIATE]

### Problem
Design an idempotent ETL pipeline that can be re-run any number of times without creating duplicate records. Use Delta Lake `MERGE INTO` to upsert data based on a primary key.

### Scenario
- Source: a batch of new/updated orders loaded each hour
- Target: a Delta table `orders_delta`
- Requirement: if the pipeline runs twice for the same batch, the target should not have duplicates
- Approach: use `order_id` as the merge key, and update existing records or insert new ones

### Key Concept
Idempotent = running the operation N times produces the same result as running it once. MERGE with a unique key ensures this because:
- Existing records are updated (not inserted again)
- New records are inserted
- Re-running with the same source data updates the same rows (no duplicates)

In [0]:
from pyspark.sql import Row
from delta.tables import DeltaTable

# Create target Delta table with initial data (using managed UC table)
spark.sql("DROP TABLE IF EXISTS day06_q10_orders")

initial_data = spark.createDataFrame([
    Row(order_id=1, customer="Alice", amount=100.0, status="shipped"),
    Row(order_id=2, customer="Bob", amount=200.0, status="pending"),
])
initial_data.write.format("delta").mode("overwrite").saveAsTable("day06_q10_orders")

# New batch of orders (includes update for order_id=1 and new order_id=3)
new_batch = spark.createDataFrame([
    Row(order_id=1, customer="Alice", amount=150.0, status="delivered"),  # UPDATE
    Row(order_id=3, customer="Carol", amount=300.0, status="pending"),     # INSERT
])

# Idempotent MERGE — can be re-run N times without duplicates
delta_table = DeltaTable.forName(spark, "day06_q10_orders")

(delta_table.alias("target")
    .merge(new_batch.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("After first MERGE:")
display(spark.table("day06_q10_orders").orderBy("order_id"))

# Re-run the same MERGE (idempotent — no duplicates, same result)
(delta_table.alias("target")
    .merge(new_batch.alias("source"), "target.order_id = source.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("After second MERGE (idempotent — same result, no duplicates):")
display(spark.table("day06_q10_orders").orderBy("order_id"))

# Verify no duplicates
count_after = spark.table("day06_q10_orders").count()
print(f"\nTotal records: {count_after} (should be 3, not 5)")
assert count_after == 3, "Idempotency failed! Duplicates detected."
print("✓ Idempotent write verified — re-running MERGE produces the same result")

# Cleanup
spark.sql("DROP TABLE IF EXISTS day06_q10_orders")

## Q11: ETL — Schema Drift Handling [INTERMEDIATE]

### Problem
Your streaming pipeline receives data with evolving schemas — new columns are added by upstream producers over time. How do you handle schema drift in Spark?

### Requirements
1. Explain how `mergeSchema` works for schema evolution
2. Show how to configure a streaming read that automatically merges new columns
3. What is the difference between `mergeSchema` and `schemaLocation`?
4. Show a code example demonstrating schema evolution with Delta Lake

### Key Concepts
- `mergeSchema=true`: Allows Delta to automatically merge new columns into the existing schema
- `schemaLocation`: Used with Auto Loader to infer and evolve schema, storing the inferred schema
- Without schema evolution: a new column causes a schema mismatch error and the job fails
- With schema evolution: new columns are added automatically (null-filled for old records)

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Step 1: Write initial data with a base schema (using managed UC table)
spark.sql("DROP TABLE IF EXISTS day06_q11_schema")

initial_data = spark.createDataFrame([
    Row(id=1, name="Alice", salary=50000),
    Row(id=2, name="Bob", salary=60000),
])
initial_data.write.format("delta").mode("overwrite").saveAsTable("day06_q11_schema")

print("Initial schema:")
spark.table("day06_q11_schema").printSchema()

# Step 2: New data with an EXTRA column (schema drift)
# Without mergeSchema, this would FAIL with schema mismatch error
new_data = spark.createDataFrame([
    Row(id=3, name="Carol", salary=70000, department="Engineering"),  # new 'department' column
    Row(id=4, name="Dave", salary=55000, department="Sales"),
])

# Write with mergeSchema=True — Delta automatically adds the new column
new_data.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("day06_q11_schema")

print("\nSchema after mergeSchema (new 'department' column added):")
result = spark.table("day06_q11_schema")
result.printSchema()
display(result.orderBy("id"))

# Old records (id=1, 2) have NULL for the new 'department' column
print("\n--- Schema Drift Handling Summary ---")
print("1. mergeSchema=true: Delta auto-adds new columns to existing schema")
print("2. Old records get NULL for newly added columns")
print("3. Auto Loader: use .option('cloudFiles.schemaLocation', path) for continuous schema inference")
print("4. Auto Loader .option('cloudFiles.schemaEvolutionMode', 'addNewColumns') for controlled evolution")
print("5. Without mergeSchema: schema mismatch causes job failure")
print("6. Best practice: use schemaLocation with Auto Loader for streaming; mergeSchema for batch")

# Cleanup
spark.sql("DROP TABLE IF EXISTS day06_q11_schema")

## Q12: Data Modeling — Star Schema Design [BASIC]

### Problem
Design a star schema for a retail sales data warehouse. Given the following business requirements:
- Track sales transactions across multiple stores
- Analyze sales by customer, product, date, and store
- Each fact row represents a single sales transaction line item

### Requirements
1. Identify the fact table and dimension tables
2. Write DDL to create: `dim_customer`, `dim_product`, `dim_store`, `dim_date`, and `fact_sales`
3. Show a sample JOIN query that aggregates sales by product category and store

### Star Schema Diagram
```
        dim_customer
             |
dim_date --- fact_sales --- dim_product
             |
          dim_store
```

In [0]:
from pyspark.sql import Row

# Create dimension tables
dim_customer = spark.createDataFrame([
    Row(customer_key=1, customer_id="C001", name="Alice", city="NYC", segment="Premium"),
    Row(customer_key=2, customer_id="C002", name="Bob", city="LA", segment="Standard"),
])

dim_product = spark.createDataFrame([
    Row(product_key=1, product_id="P001", name="Laptop", category="Electronics", price=999.99),
    Row(product_key=2, product_id="P002", name="Coffee Mug", category="Home", price=9.99),
])

dim_store = spark.createDataFrame([
    Row(store_key=1, store_id="S001", name="Downtown", region="East"),
    Row(store_key=2, store_id="S002", name="Mall", region="West"),
])

dim_date = spark.createDataFrame([
    Row(date_key=20260101, date="2026-01-01", month="January", quarter="Q1", year=2026),
    Row(date_key=20260102, date="2026-01-02", month="January", quarter="Q1", year=2026),
])

# Create fact table — grain = one row per sales line item
fact_sales = spark.createDataFrame([
    Row(sales_key=1, date_key=20260101, customer_key=1, product_key=1, store_key=1, quantity=2, amount=1999.98),
    Row(sales_key=2, date_key=20260101, customer_key=2, product_key=2, store_key=2, quantity=5, amount=49.95),
    Row(sales_key=3, date_key=20260102, customer_key=1, product_key=2, store_key=1, quantity=10, amount=99.90),
])

# Register as temp views for SQL queries
for name, df in [("dim_customer", dim_customer), ("dim_product", dim_product), 
                  ("dim_store", dim_store), ("dim_date", dim_date), ("fact_sales", fact_sales)]:
    df.createOrReplaceTempView(name)

print("Fact table:")
display(fact_sales)

# Star schema JOIN: aggregate sales by product category and store
result = spark.sql("""
    SELECT 
        p.category AS product_category,
        s.name AS store_name,
        d.month,
        SUM(f.quantity) AS total_quantity,
        ROUND(SUM(f.amount), 2) AS total_revenue
    FROM fact_sales f
    JOIN dim_product p ON f.product_key = p.product_key
    JOIN dim_store s ON f.store_key = s.store_key
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY p.category, s.name, d.month
    ORDER BY total_revenue DESC
""")

print("\nStar schema JOIN — Sales by category, store, month:")
display(result)

print("\n--- Star Schema Principles ---")
print("1. Fact table: contains measures (quantitative data) + foreign keys to dimensions")
print("2. Dimension tables: contain descriptive attributes (no measures)")
print("3. Fact table grain: the level of detail — here, one row per sales line item")
print("4. Foreign keys in fact → primary keys in dimensions")
print("5. Star schema: fact at center, dimensions radiate out (denormalized for query speed)")

## Q13: Data Modeling — SCD Type 2 with MERGE [INTERMEDIATE]

### Problem
Implement Slowly Changing Dimension Type 2 (SCD Type 2) for a `dim_customer_scd2` table using Delta Lake MERGE.

### Requirements
When a customer's attribute (e.g., `city`) changes:
1. Expire the old record: set `end_date` to current date, `is_current` to false
2. Insert a new record: with `start_date` = current date, `end_date` = NULL, `is_current` = true
3. Use a surrogate key (`customer_sk`) for uniqueness
4. Preserve the history of all changes

### Sample Data
| customer_sk | customer_id | name | city | start_date | end_date | is_current |
|--------------|-------------|------|------|------------|----------|-----------|
| 1 | C001 | Alice | NYC | 2026-01-01 | NULL | true |
| 2 | C002 | Bob | LA | 2026-01-01 | NULL | true |

In [0]:
from pyspark.sql import Row
from delta.tables import DeltaTable
from pyspark.sql.functions import lit, current_date, col, row_number
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

# Create initial SCD Type 2 table (using managed UC table)
spark.sql("DROP TABLE IF EXISTS day06_q13_scd2")

scd2_schema = StructType([
    StructField("customer_sk", LongType(), True),
    StructField("customer_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("start_date", StringType(), True),
    StructField("end_date", StringType(), True),
    StructField("is_current", BooleanType(), True),
])

initial = spark.createDataFrame([
    (1, "C001", "Alice", "NYC", "2026-01-01", None, True),
    (2, "C002", "Bob", "LA", "2026-01-01", None, True),
], schema=scd2_schema)
initial.write.format("delta").mode("overwrite").saveAsTable("day06_q13_scd2")

# Incoming update: Alice moved from NYC to Boston; new customer Carol
updates = spark.createDataFrame([
    Row(customer_id="C001", name="Alice", city="Boston"),   # CHANGED attribute
    Row(customer_id="C002", name="Bob", city="LA"),          # NO CHANGE (should not update)
    Row(customer_id="C003", name="Carol", city="SF"),        # NEW customer
])

delta_table = DeltaTable.forName(spark, "day06_q13_scd2")

# Step 1: Expire records where attributes changed (UPDATE existing current records)
delta_table.alias("target").merge(
    updates.alias("src"),
    "target.customer_id = src.customer_id AND target.is_current = true"
).whenMatchedUpdate(
    condition="target.city != src.city",
    set={
        "end_date": lit("2026-01-15"),
        "is_current": lit(False)
    }
).execute()

# Step 2: Insert new current records for changed + new customers
# Get max customer_sk for surrogate key
max_sk = spark.table("day06_q13_scd2").agg({"customer_sk": "max"}).collect()[0][0]

new_records = updates.alias("src").join(
    spark.table("day06_q13_scd2").alias("tgt"),
    (col("src.customer_id") == col("tgt.customer_id")) & (col("tgt.is_current") == True),
    "left_anti"
).select(
    (lit(max_sk) + lit(1)).alias("sk_start"),
    col("src.customer_id"),
    col("src.name"),
    col("src.city"),
    lit("2026-01-15").alias("start_date"),
    lit(None).cast("string").alias("end_date"),
    lit(True).alias("is_current")
)

# Add sequential SK
from pyspark.sql.functions import row_number, monotonically_increasing_id
w = Window.orderBy(lit(1))
new_records = new_records.withColumn("customer_sk", lit(max_sk) + row_number().over(w))
new_records = new_records.select("customer_sk", "customer_id", "name", "city", "start_date", "end_date", "is_current")

new_records.write.format("delta").mode("append").saveAsTable("day06_q13_scd2")

# Show the full history
result = spark.table("day06_q13_scd2").orderBy("customer_id", "start_date")
print("SCD Type 2 — Full customer history:")
display(result)

print("\n--- SCD Type 2 Summary ---")
print("1. When an attribute changes: expire old record (end_date + is_current=false), insert new record")
print("2. History is preserved — you can query 'what was Alice's city on date X'")
print("3. Surrogate key (customer_sk) uniquely identifies each version")
print("4. Use is_current=true to get the latest version of each customer")
print("5. MERGE with condition ensures only changed records are expired")

# Cleanup
spark.sql("DROP TABLE IF EXISTS day06_q13_scd2")

## Q14: Snowflake — Micro-partitions [BASIC]

### Problem
Explain Snowflake's micro-partition architecture:
1. What is a micro-partition? How is it different from Hive-style partitioning?
2. How does Snowflake prune micro-partitions during query execution?
3. What are clustering keys and how do they affect performance?
4. Write SQL examples to create a table with a clustering key and query with pruning.

### Key Concepts
- **Micro-partition**: Snowflake stores data in ~50-500MB compressed units (much smaller than traditional partitions)
- **Automatic pruning**: Snowflake tracks min/max values per column per micro-partition; queries skip irrelevant partitions
- **Clustering keys**: Define sort order within micro-partitions for better pruning (important for large tables)
- **No manual partitioning**: Unlike Hive, Snowflake manages partitioning automatically

In [0]:
# Snowflake SQL examples (demonstration — run in Snowflake, not Spark)
# These are explanatory SQL statements for Snowflake environment

snowflake_sql = """
-- 1. Create a table with a clustering key (e.g., on 'order_date')
CREATE TABLE orders (
    order_id INT,
    customer_id INT,
    order_date DATE,
    amount DECIMAL(10,2),
    status STRING
)
CLUSTER BY (order_date);

-- 2. Insert data (Snowflake automatically creates micro-partitions)
INSERT INTO orders VALUES
(1, 101, '2026-01-01', 100.00, 'shipped'),
(2, 102, '2026-01-02', 200.00, 'pending'),
(3, 103, '2026-01-15', 150.00, 'shipped'),
(4, 104, '2026-02-01', 300.00, 'delivered'),
(5, 105, '2026-02-15', 250.00, 'shipped');

-- 3. Query with date filter — Snowflake prunes micro-partitions automatically
-- Only micro-partitions containing '2026-01-01' to '2026-01-31' are scanned
SELECT * FROM orders 
WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31';

-- 4. Check clustering information
SELECT SYSTEM$CLUSTERING_INFORMATION('orders', '(order_date)');

-- 5. View micro-partition stats
DESC TABLE orders;
"""

print(snowflake_sql)

print("""
--- Micro-partition Architecture Summary ---

1. SIZE: Each micro-partition is 50-500MB (compressed). A 1TB table = ~2000-20000 micro-partitions.

2. AUTO PRUNING: Snowflake maintains metadata for each micro-partition:
   - Min/max values for each column
   - Number of rows, NULL count, distinct count
   Query: SELECT * FROM orders WHERE order_date = '2026-01-15'
   → Snowflake checks min/max of order_date per micro-partition
   → Only scans partitions where 2026-01-15 falls within [min, max]
   → Prunes (skips) all other partitions — no full scan!

3. CLUSTERING KEYS:
   - CLUSTER BY (order_date) → data is sorted by order_date within micro-partitions
   - Better clustering = better pruning = faster queries
   - Auto-clustering maintains clustering as new data arrives
   - Use clustering keys on columns frequently used in WHERE/JOIN/GROUP BY

4. VS HIVE PARTITIONING:
   - Hive: manual partition directories (partition_prdt=Electronics/), coarse-grained
   - Snowflake: automatic, fine-grained, metadata-driven, no manual management
   - Hive: partition pruning uses directory structure; Snowflake: uses column min/max metadata
""")

## Q15: Snowflake — Time Travel and UNDROP [INTERMEDIATE]

### Problem
Demonstrate Snowflake's Time Travel feature:
1. Query data at a specific point in time using `AT(TIMESTAMP => ...)`
2. Query data from a previous version using `AT(STATEMENT => ...)` or `BEFORE(STATEMENT => ...)`
3. Restore a dropped table using `UNDROP TABLE`
4. Set and configure the data retention period
5. Explain the relationship between Time Travel, Fail-safe, and data retention costs

### Key Concepts
- **Time Travel**: Query or restore data from the past (up to the retention period)
- **Standard Edition**: 1 day (max 1 day)
- **Enterprise Edition**: 0-90 days (configurable per table/object)
- **Fail-safe**: 7-day additional recovery (Snowflake-managed, after Time Travel expires)
- **UNDROP**: Restore a dropped table (within Time Travel retention period)

In [0]:
# Snowflake SQL examples (demonstration — run in Snowflake, not Spark)

snowflake_time_travel_sql = """
-- 1. Set data retention period for a table (Enterprise Edition: up to 90 days)
CREATE TABLE orders (
    order_id INT,
    amount DECIMAL(10,2),
    status STRING
) DATA_RETENTION_TIME_IN_DAYS = 30;

-- 2. Insert initial data
INSERT INTO orders VALUES (1, 100.00, 'pending'), (2, 200.00, 'shipped');

-- 3. Update a record (this creates a new version of the table)
UPDATE orders SET status = 'delivered' WHERE order_id = 1;

-- 4. Time Travel: query data 5 minutes ago
SELECT * FROM orders AT(TIMESTAMP => DATEADD(minute, -5, CURRENT_TIMESTAMP()));

-- 5. Time Travel: query data at a specific timestamp
SELECT * FROM orders AT(OFFSET => -60 * 5);  -- 5 minutes ago

-- 6. Time Travel: query BEFORE a specific statement (using query ID)
-- Find the query ID of the UPDATE statement
SELECT query_id FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY()) 
WHERE query_text LIKE '%UPDATE orders%' ORDER BY start_time DESC LIMIT 1;

-- Query the data BEFORE the UPDATE ran
SELECT * FROM orders BEFORE(STATEMENT => '<query_id_from_above>');

-- 7. Restore data from Time Travel (undo the UPDATE)
CREATE TABLE orders_restored AS 
SELECT * FROM orders BEFORE(STATEMENT => '<query_id_from_above>');

-- 8. UNDROP: restore a dropped table
DROP TABLE orders;
-- Oops! Restore it:
UNDROP TABLE orders;

-- 9. View table history
SELECT * FROM TABLE(GET_REFERENCES('orders'));

-- 10. Set retention at database/schema level
ALTER DATABASE my_db SET DATA_RETENTION_TIME_IN_DAYS = 60;
ALTER SCHEMA my_db.my_schema SET DATA_RETENTION_TIME_IN_DAYS = 45;
"""

print(snowflake_time_travel_sql)

print("""
--- Time Travel & UNDROP Summary ---

1. AT(TIMESTAMP => ...): Query data as it existed at a specific timestamp
   AT(OFFSET => -N): N seconds ago
   BEFORE(STATEMENT => query_id): Data before a specific statement ran

2. UNDROP TABLE: Restores a dropped table (within retention period)
   - Works for tables, schemas, and databases
   - Must be done within the data retention period
   - Faster than recreating from backup

3. RETENTION PERIODS:
   - Standard Edition: 0-1 day (default 1 day)
   - Enterprise Edition: 0-90 days (default 1 day)
   - Set per-object: DATA_RETENTION_TIME_IN_DAYS = N
   - Can be set at database, schema, or table level

4. FAIL-SAFE vs TIME TRAVEL:
   - Time Travel: user-accessible (query/restore past data) — configurable period
   - Fail-safe: Snowflake-managed recovery (7 days after Time Travel expires)
   - Fail-safe is for disaster recovery, requires Snowflake support to access
   - Both consume storage costs — longer retention = more storage

5. COST IMPLICATIONS:
   - Time Travel data = same storage cost as current data
   - Fail-safe data = also billed
   - Cloning (zero-copy) is free for storage — use clones for testing
""")

## Q16: Monitoring — Databricks Jobs Task Types and Run If Dependencies [BASIC]

### Problem
Explain the Databricks Jobs orchestration capabilities:
1. List the different task types available in Databricks Jobs (notebook, SQL, Python script, etc.)
2. Explain how `Run If` dependencies work for conditional task execution
3. Compare Databricks Jobs with Apache Airflow and Azure Data Factory (ADF)
4. Design a simple job pipeline: Extract → Transform → Load → Notify, where the Notify task runs only if the Load task succeeds, and a Retry task runs if it fails

### Key Concepts
- **Task Types**: Notebook, SQL query, Python script, Spark JAR, Delta Live Tables pipeline, Run If, dbt task, Wheel, etc.
- **Dependencies**: Tasks can depend on other tasks; `Run If` conditionals control execution flow
- **Run If conditions**: `all_success` (default), `all_failed`, `at_least_one_success`, `none_failed`, `always`
- **Retries**: Configurable per-task retry count and retry interval

In [0]:
# Databricks Jobs orchestration — explanatory (JSON job config example)

job_config = """
{
  "name": "ETL Pipeline — Extract/Transform/Load/Notify",
  "tasks": [
    {
      "task_key": "extract",
      "notebook_task": {
        "notebook_path": "/pipelines/extract_data",
        "source": "WORKSPACE"
      },
      "max_retries": 3,
      "retry_on_timeout": true,
      "min_retry_interval_millis": 60000
    },
    {
      "task_key": "transform",
      "depends_on": [{"task_key": "extract"}],
      "notebook_task": {
        "notebook_path": "/pipelines/transform_data"
      },
      "max_retries": 2
    },
    {
      "task_key": "load",
      "depends_on": [{"task_key": "transform"}],
      "sql_task": {
        "query": {
          "query_id": "load-query-uuid"
        }
      }
    },
    {
      "task_key": "notify_success",
      "depends_on": [{"task_key": "load", "type": "all_success"}],
      "notebook_task": {
        "notebook_path": "/pipelines/send_notification",
        "base_parameters": {"status": "success"}
      }
    },
    {
      "task_key": "notify_failure",
      "depends_on": [{"task_key": "load", "type": "at_least_one_failed"}],
      "notebook_task": {
        "notebook_path": "/pipelines/send_alert",
        "base_parameters": {"status": "failure"}
      }
    }
  ],
  "schedule": {
    "quartz_cron_expression": "0 0 9 * * ?",
    "timezone_id": "Asia/Kolkata"
  }
}
"""

print(job_config)

print("""
--- Databricks Jobs Orchestration Summary ---

1. TASK TYPES:
   - Notebook: Run a Databricks notebook
   - SQL: Run a SQL query or file
   - Python script: Run a .py file
   - Spark JAR: Run a compiled Java/Scala JAR
   - Delta Live Tables: Run an SDP pipeline
   - dbt: Run a dbt project
   - Wheel: Run a Python wheel package
   - Run If: Conditional branching (no execution, just dependency)

2. RUN IF DEPENDENCY TYPES:
   - all_success (default): Run only if ALL upstream tasks succeeded
   - all_failed: Run only if ALL upstream tasks failed
   - at_least_one_success: Run if at least one upstream succeeded
   - none_failed: Run if no upstream failed (all succeeded or were skipped)
   - always: Run regardless of upstream results (cleanup/notify)

3. COMPARISON: Databricks Jobs vs Airflow vs ADF
   | Feature          | Databricks Jobs          | Airflow              | ADF                  |
   |-----------------|-------------------------|---------------------|---------------------|
   | Native Spark     | Yes (built-in)           | No (needs operator)  | Via linked service   |
   | Language         | Python/SQL/Scala        | Python (DAGs)       | JSON/visual           |
   | UI               | Databricks workspace    | Airflow UI          | Azure portal         |
   | Retry            | Per-task config         | Per-task config     | Per-activity config  |
   | Scheduling       | Cron + file/table trigger| Cron/sensor         | Schedule/event       |
   | Best for         | Spark/data pipelines    | General orchestration | Azure ecosystem      |

4. TRIGGERS:
   - Cron schedule: e.g., 0 0 9 * * ? (daily at 9 AM)
   - File arrival: trigger when file lands in cloud storage
   - Table update: trigger when a Delta table is updated
""")

## Q17: Monitoring — Data Freshness SQL Alerts [INTERMEDIATE]

### Problem
Design a data freshness monitoring SQL alert for a Delta Lake table:
1. Write a SQL query that checks whether the `fact_sales` table has been updated within the last expected interval (e.g., 6 hours)
2. Describe how to set up a Databricks SQL Alert that triggers when the freshness check fails
3. Include a query that also reports row count and data volume metrics
4. Explain the metrics-table pattern: writing pipeline run metrics to a dedicated monitoring table

### Monitoring Strategy
- **Pipeline-level**: Job status, run duration, task failures (from `system.lakeflow.*`)
- **Data-level**: Freshness (last update time), volume (row count), quality (null %, duplicate %)
- **Alerts**: Databricks SQL Alerts evaluate a query on a schedule and notify when a threshold is breached

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, max as _max, count as _count, current_timestamp, expr

# Simulate a fact_sales table with a last_updated column
fact_data = spark.createDataFrame([
    Row(order_id=1, amount=100.0, updated_at="2026-10-18 06:00:00"),
    Row(order_id=2, amount=200.0, updated_at="2026-10-18 06:00:00"),
    Row(order_id=3, amount=150.0, updated_at="2026-10-18 07:30:00"),
])
fact_data.createOrReplaceTempView("fact_sales")

# --- 1. Freshness check query ---
# This query returns 1 if the table is stale (no update in last 6 hours), 0 if fresh
freshness_query = """
SELECT 
    CASE 
        WHEN MAX(CAST(updated_at AS TIMESTAMP)) < CURRENT_TIMESTAMP() - INTERVAL 6 HOURS THEN 1
        ELSE 0
    END AS is_stale,
    MAX(updated_at) AS last_update_time,
    CURRENT_TIMESTAMP() AS check_time,
    BIGINT(TIMESTAMPDIFF(HOUR, MAX(CAST(updated_at AS TIMESTAMP)), CURRENT_TIMESTAMP())) AS hours_since_update
FROM fact_sales
"""
freshness_result = spark.sql(freshness_query)
print("1. Freshness check:")
display(freshness_result)

# --- 2. Data volume + quality metrics query ---
metrics_query = """
SELECT 
    'fact_sales' AS table_name,
    COUNT(*) AS row_count,
    MAX(updated_at) AS last_update_time,
    COUNT(DISTINCT order_id) AS distinct_orders,
    ROUND(COUNT(DISTINCT order_id) * 100.0 / COUNT(*), 2) AS uniqueness_pct,
    SUM(CASE WHEN amount IS NULL THEN 1 ELSE 0 END) AS null_amount_count,
    CURRENT_TIMESTAMP() AS check_timestamp
FROM fact_sales
"""
metrics_result = spark.sql(metrics_query)
print("2. Data volume & quality metrics:")
display(metrics_result)

# --- 3. Metrics table pattern: log metrics for tracking ---
print("3. Metrics table pattern:")
metrics_result.write.mode("append").saveAsTable("default.pipeline_metrics")
display(spark.sql("SELECT * FROM default.pipeline_metrics ORDER BY check_timestamp DESC"))

print("""
--- Setting Up Databricks SQL Alerts ---

1. CREATE SQL QUERY: Save the freshness query above as a Databricks SQL Query
2. CREATE ALERT:
   - Query: the freshness check query (returns is_stale column)
   - Trigger: WHEN is_stale = 1 (table is stale)
   - Schedule: Every 1 hour (or appropriate interval)
   - Notification: Email, Slack, PagerDuty, or webhook

3. SQL ALERT CONFIGURATION:
   - Trigger condition: column 'is_stale' > 0
   - Comparison: GREATER_THAN 0
   - Frequency: Every 60 minutes
   - Destinations: Email + Slack webhook

4. METRICS TABLE PATTERN:
   - Create a monitoring table (e.g., pipeline_metrics) to log run metrics
   - Each pipeline run appends: table_name, row_count, last_update_time, null_counts, etc.
   - Query this table to track trends, detect anomalies, build dashboards
   - Example: SELECT table_name, AVG(row_count) FROM pipeline_metrics GROUP BY table_name

5. SYSTEM TABLES:
   - system.lakeflow.jobs: job run history, duration, status
   - system.lakeflow.job_runs: detailed task-level run info
   - system.billing.usage: cost monitoring
""")

## Q18: Monitoring — Logging Best Practices [INTERMEDIATE]

### Problem
Compare Python's `logging` module vs `print()` statements for ETL pipelines and explain best practices:

1. Why is `logging` better than `print()` for production pipelines?
2. Show how to configure structured logging with log levels (DEBUG, INFO, WARNING, ERROR, CRITICAL)
3. What information should be logged at each stage of an ETL pipeline (extract → transform → load)?
4. How to integrate Python logging with Spark log4j and Databricks driver logs?
5. Debugging walkthrough: a pipeline fails at 3 AM — what logs do you check and in what order?

### Log Levels Hierarchy
```
DEBUG (10) → INFO (20) → WARNING (30) → ERROR (40) → CRITICAL (50)
```

In [0]:
import logging
import json
from datetime import datetime

# --- 1. Configure structured logging ---
logger = logging.getLogger("etl_pipeline")
logger.setLevel(logging.DEBUG)

# Console handler with structured format
handler = logging.StreamHandler()
formatter = logging.Formatter(
    '%(asctime)s | %(levelname)-8s | %(name)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
handler.setFormatter(formatter)
logger.addHandler(handler)

# --- 2. Simulate ETL pipeline with structured logging ---

def log_metric(stage, metric_name, value):
    """Log a structured metric for monitoring."""
    logger.info(json.dumps({
        "stage": stage,
        "metric": metric_name,
        "value": value,
        "timestamp": datetime.now().isoformat()
    }))

def extract_data():
    logger.info("EXTRACT: Starting data extraction from source")
    logger.debug("EXTRACT: Connecting to source database")
    row_count = 10000
    logger.info(f"EXTRACT: Successfully extracted {row_count} rows")
    log_metric("extract", "rows_extracted", row_count)
    return list(range(row_count))

def transform_data(data):
    logger.info("TRANSFORM: Starting data transformation")
    logger.debug(f"TRANSFORM: Processing {len(data)} rows")
    
    # Simulate a data quality issue
    null_count = 5
    if null_count > 0:
        logger.warning(f"TRANSFORM: Found {null_count} NULL values — applying default")
    
    transformed = [x * 2 for x in data]
    logger.info(f"TRANSFORM: Completed — {len(transformed)} rows transformed")
    log_metric("transform", "rows_transformed", len(transformed))
    return transformed

def load_data(data):
    logger.info("LOAD: Starting data load to Delta table")
    try:
        logger.debug(f"LOAD: Writing {len(data)} rows to Delta table")
        # Simulate successful write
        logger.info(f"LOAD: Successfully loaded {len(data)} rows to Delta table")
        log_metric("load", "rows_loaded", len(data))
        return True
    except Exception as e:
        logger.error(f"LOAD: Failed to write data — {str(e)}", exc_info=True)
        return False

# Run the pipeline
logger.info("=== ETL Pipeline Started ===")
data = extract_data()
transformed = transform_data(data)
success = load_data(transformed)

if success:
    logger.info("=== ETL Pipeline Completed Successfully ===")
else:
    logger.critical("=== ETL Pipeline FAILED ===")

print("""
--- Logging Best Practices Summary ---

1. logging vs print():
   - print(): No log levels, no formatting, goes to stdout, can't filter, can't redirect
   - logging: Log levels (DEBUG/INFO/WARNING/ERROR/CRITICAL), structured output, 
     can filter by level, can send to files/console/webhooks, integrates with Spark log4j

2. WHAT TO LOG AT EACH STAGE:
   - EXTRACT: source connection info, row count, source schema, extraction duration
   - TRANSFORM: rows processed, null counts, duplicate counts, validation failures, 
     schema changes, transformation duration
   - LOAD: target table name, rows written, upserts vs inserts, write duration, 
     partition info

3. 3 AM FAILURE DEBUGGING WALKTHROUGH (in order):
   Step 1: Check Databricks Job run history — which task failed? What's the error?
   Step 2: Check the job cluster logs — driver logs (stderr/stdout) for exceptions
   Step 3: Check Spark UI — failed stages, OOM errors, data skew
   Step 4: Check Delta table history — DESCRIBE HISTORY for the target table
   Step 5: Check data quality — was the source data delayed or malformed?
   Step 6: Check system tables — system.lakeflow.job_runs for detailed run info
   Step 7: Reproduce locally with a sample of the failed data batch

4. STRUCTURED LOGGING:
   - Log JSON-formatted metrics for easy parsing by monitoring tools
   - Include: timestamp, stage, metric_name, value, run_id
   - Send to a central log aggregator (Datadog, Splunk, CloudWatch)

5. SPARK LOG4J INTEGRATION:
   - Spark uses log4j for JVM logs; Python logging is separate
   - Set log level: spark.sparkContext.setLogLevel('WARN')
   - Driver logs available in Databricks UI → Cluster → Logs → Driver
   - Use Databricks logging: dbutils.log.append() for structured event logging
""")

## 🔐 Section 9: Production Pillars — Security & Backfills (Project-Based) (2 Questions)

---
### Q19: Security in MY Pipelines — FraudStream + Tesla [INTERMEDIATE]

**Scenario**: Interviewer asks: "Your fraud pipeline has card numbers and your telemetry pipeline has driver VIN/phone. How is YOUR pipeline secured — access, credentials, encryption, and PII?"

**Question**: Answer using your actual projects. Cover 4 layers: **access control** (Unity Catalog grants — who reads bronze vs gold), **credentials/secrets** (how the Snowflake sync gets its password), **encryption** (S3 at rest, TLS in transit), and **PII handling** (card numbers in FraudStream, VIN/driver data in Tesla). Frame every answer as "in my pipeline, ...".

**Hint**: Study order: UC grants (catalog→schema→table) → secret scopes + dbutils.secrets → S3 SSE-KMS → hash/tokenize at ingestion + column masks. Interview bolne ka order: "4 layers — access, secrets, encryption, PII — and in my projects..."

In [0]:
# SOLUTION Q19

"""
SECURITY — MODEL ANSWER IN "MY PROJECT" STYLE
(Study this as a story you tell about YOUR pipelines, not as a list)

INTERVIEW OPENING LINE:
  "I think of pipeline security in 4 layers — access control, secrets,
   encryption, and PII handling. In my projects, FraudStream and Tesla
   telemetry, each layer looks like this..."

LAYER 1 — ACCESS CONTROL (Unity Catalog):
  1-line: least privilege per layer, via UC grants.
  Detail: catalog → schema → table hierarchy; pipelines run as a SERVICE
    PRINCIPAL, not my user.
  MY PROJECT: bronze (raw) schemas readable only by the pipeline principal —
    analysts get SELECT on gold only (silver/silver-adjacent tables stay
    private because raw PII lives there). FraudStream alerts table readable
    by the fraud-ops team; Tesla gold by BI.
  Key syntax to remember:
      GRANT SELECT ON SCHEMA gold TO `analysts`;
      GRANT WRITE ON SCHEMA silver TO `pipeline-runner`;

LAYER 2 — SECRETS / CREDENTIALS:
  1-line: zero hardcoded credentials — everything from secret scopes.
  Detail: dbutils.secrets.get(scope, key) at runtime; scope backed by
    AWS Secrets Manager; rotation without code change.
  MY PROJECT: the Tesla gold → Snowflake sync job reads the Snowflake
    password/warehouse creds from a secret scope — the notebook has
    NO password in it, so it's safe to commit to git.
      snowflake_pwd = dbutils.secrets.get(scope='prod', key='snowflake-pwd')
  S3 access: UC external locations + storage credentials (instance
    profile) — no access keys anywhere.

LAYER 3 — ENCRYPTION:
  1-line: encrypted at rest (S3 SSE-KMS) and in transit (TLS), by default.
  Detail: Delta tables on S3 inherit bucket-level encryption; Snowflake
    side supports customer-managed keys (Tri-Secret Secure) if compliance
    needs it; JDBC/Kafka connections over TLS.
  MY PROJECT: both pipelines read/write S3 buckets with SSE-KMS enabled;
    the Snowflake JDBC URL uses SSL.

LAYER 4 — PII HANDLING (the layer interviewers probe hardest):
  1-line: classify early, hash/tokenize at ingestion, mask at query time.
  MY PROJECT (FraudStream): card numbers → HASHED at ingestion
    (tokenization; store only last-4 + hash for joins). The alerts table
    never carries a raw PAN. Fraud-ops sees masked output; only the masked
    view is exposed to dashboards.
  MY PROJECT (Tesla): driver phone/address → column masking policies;
    VIN → hashed for joins in silver, full VIN only where strictly needed.
    Logs/alerts NEVER print raw PII — I scrub payloads before logging.
  Column mask syntax to remember:
      CREATE FUNCTION mask_phone(phone STRING) RETURN CASE
        WHEN is_account_group_member('fraud-team') THEN phone
        ELSE CONCAT('***-***-', RIGHT(phone, 4)) END;
      ALTER TABLE gold.driver_profiles
        ALTER COLUMN phone SET MASKING POLICY mask_phone;

FOLLOW-UPS INTERVIEWERS ASK (be ready):
  - "Where do you store secrets?" → secret scopes backed by AWS Secrets
    Manager; nothing in notebooks or DAB files.
  - "What if a PII column sneaks into bronze?" → UC sensitive-data
    classification flags it; alert + move/hash it in the next silver run.
  - "Who can see raw card data?" → almost nobody: hashed at ingestion,
    so NOBODY re-derives PAN — even I can't.

STUDY TIP:
  Don't memorize this as 4 topics. Memorize ONE sentence per layer + one
  project example per layer. In the interview, each layer takes ~30 sec:
  "In my pipeline, X is handled by Y, for example Z."
"""

print("See comments for the full project-based explanation.")

---
### Q20: Backfilling MY Tesla Pipeline — Silver Bug, 90 Bad Partitions [INTERMEDIATE]

**Scenario**: Interviewer asks: "A bug in your silver transformation was found after 3 months — 90 daily partitions of `silver.enriched_telemetry` have wrong battery-health math. The daily job must keep running. What do you do?"

**Question**: Walk through the backfill for YOUR pipeline: the **idempotent partition-overwrite pattern** (`replaceWhere` / `INSERT OVERWRITE ... PARTITION`), how you **isolate** the backfill from the daily schedule, how you **validate** before cutover, and how you **control cost**. Answer as "in my Tesla pipeline, ..." — a story, not a checklist.

**Hint**: Study order: fix-forward → targeted overwrite (idempotent) → separate job → staging compare → time travel as safety net. Interview bolne ka order: "First I'd fix the code, then recompute history surgically without touching the daily schedule..."

In [0]:
# SOLUTION Q20

"""
BACKFILL — MODEL ANSWER IN "MY PROJECT" STYLE
(A 90-second story about YOUR Tesla pipeline)

INTERVIEW OPENING LINE:
  "First I'd fix the transformation code forward — then recompute history
   surgically, without stopping the daily job. In my Tesla pipeline it
   would look like this..."

STEP 1 — FIX FORWARD FIRST:
  - Confirm the bug really affects gold too (does battery-health roll up
    into gold KPIs? if yes, gold needs recompute as well).
  - Fix the silver transformation, unit-test it on ONE known bad day.

STEP 2 — SURGICAL, IDEMPOTENT OVERWRITE (the core pattern):
  - My silver table is partitioned by event_date, so I overwrite ONLY the
    bad date range — never drop/recreate the table:

      (fixed_df.writeTo('silver.enriched_telemetry')
           .replaceWhere('event_date BETWEEN "2026-07-20" AND "2026-10-17"')
           .overwrite())

      -- or, per-day in SQL:
      INSERT OVERWRITE silver.enriched_telemetry PARTITION (event_date = '2026-08-01')
      SELECT ... FROM bronze.raw_telemetry WHERE event_date = '2026-08-01';

  - WHY IDEMPOTENT (say this line): the overwrite atomically replaces those
    partitions — if the backfill crashes at day 45 and I re-run, days 1-44
    are rewritten with the SAME data. No duplicates, no double-processing.
    That's the difference between a backfill plan and a resume-from-hell.

STEP 3 — ISOLATE FROM THE DAILY JOB:
  - Daily job keeps its 2 AM schedule untouched — it only writes TODAY's
    partition anyway, so no conflict (Delta optimistic concurrency handles
    different files in the same table).
  - Backfill runs as a SEPARATE one-time job on a bigger cluster.
  - My daily notebook already takes start_date/end_date parameters, so the
    SAME notebook does both: daily run (today) and backfill (90-day range).
    One code path, two use cases — nothing new to test.

STEP 4 — VALIDATE BEFORE CUTOVER:
  - Compute the corrected 90 days into a STAGING table first, then compare:
      SELECT * FROM silver.enriched_telemetry
      WHERE event_date BETWEEN '2026-07-20' AND '2026-10-17'
      EXCEPT SELECT * FROM silver_backfill_compare;  -- should show ONLY expected diffs
  - Check aggregates on a few known vehicles: SUM(battery_health), row counts.
  - Snapshot the table before overwriting (DESCRIBE HISTORY) → if my "fix"
    makes things worse, time travel restores in one command.

STEP 5 — COST CONTROL:
  - Split into 3 jobs of 30 days (better retry granularity, less spill).
  - Reuse bronze raw data — never re-ingest from S3.
  - Autoscale + auto-terminate the backfill cluster.

FOLLOW-UPS INTERVIEWERS ASK (be ready):
  - "Why not just re-run the daily job 90 times?" → same thing at small
    scale, but a parameterized backfill job is faster, parallel, and has
    one validation point instead of 90.
  - "What about late-arriving data during the backfill?" → the daily job
    still appends today's partition; if a historical file lands, the
    next daily run picks it up and I re-run that one partition.
  - "How do you prevent this next time?" → unit tests on transformations
    + metrics table with per-partition row counts (drift = early warning)
    + DQ checks per run.

STUDY TIP:
  The whole answer is ONE sentence with 5 steps: fix → surgically
  overwrite (idempotent) → separate job → validate vs staging → control
  cost. Practice saying it out loud in 90 seconds with the replaceWhere
  syntax as your "I've actually done this" proof.
"""

print("See comments for the full project-based explanation.")

# Day 06 — Score Card

| # | Category | Question | Difficulty | Solved? | Notes |
|---|----------|----------|------------|---------|-------|
| 1 | SQL Advanced | Q1: ROW_NUMBER Deduplication | BASIC | ☐ | |
| 2 | SQL Advanced | Q2: Cumulative Sum / Running Total | INTERMEDIATE | ☐ | |
| 3 | Python | Q3: Two Sum | BASIC | ☐ | |
| 4 | Python | Q4: Flatten Nested List | INTERMEDIATE | ☐ | |
| 5 | PySpark | Q5: Filter, Select, Aggregate | BASIC | ☐ | |
| 6 | PySpark | Q6: Multiple Join Types | INTERMEDIATE | ☐ | |
| 7 | Spark Opt | Q7: Repartition vs Coalesce | BASIC | ☐ | |
| 8 | Spark Opt | Q8: Parquet vs Delta Lake | INTERMEDIATE | ☐ | |
| 9 | ETL | Q9: Structured Streaming Checkpointing | BASIC | ☐ | |
| 10 | ETL | Q10: Idempotent Writes with MERGE | INTERMEDIATE | ☐ | |
| 11 | ETL | Q11: Schema Drift Handling | INTERMEDIATE | ☐ | |
| 12 | Data Modeling | Q12: Star Schema Design | BASIC | ☐ | |
| 13 | Data Modeling | Q13: SCD Type 2 with MERGE | INTERMEDIATE | ☐ | |
| 14 | Snowflake | Q14: Micro-partitions | BASIC | ☐ | |
| 15 | Snowflake | Q15: Time Travel & UNDROP | INTERMEDIATE | ☐ | |
| 16 | Monitoring | Q16: Jobs Task Types & Run If | BASIC | ☐ | |
| 17 | Monitoring | Q17: Data Freshness SQL Alerts | INTERMEDIATE | ☐ | |
| 18 | Monitoring | Q18: Logging Best Practices | INTERMEDIATE | ☐ | |
| 19 | Production Pillars | Q19: Security in My Pipelines (Project) | INTERMEDIATE | ☐ | |
| 20 | Production Pillars | Q20: Backfilling My Tesla Pipeline (Project) | INTERMEDIATE | ☐ | |

---

## Day 06 Summary
- **Total Questions**: 20
- **Categories**: 9
- **Difficulty Level**: BASIC → INTERMEDIATE (Week 1, Day 6)
- **Time Spent**: ___ minutes
- **Score**: ___ / 20
- **Key Takeaways**: 
  - Practice writing for each problem before looking at solutions
  - Time yourself: aim for 15-20 minutes per question
  - Review solutions carefully and understand the approach
  - Re-attempt questions you got wrong after 2 days